In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import time
from multiprocessing import Pool
from tqdm.auto import tqdm
import re
from copy import deepcopy

import numpy as np
from scipy import integrate
from matplotlib import pyplot as plt

import noctiluca as nl
import bayesmsd

/home/sgh/gitlibs/chromatin_dynamics/.venv_SD_py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
filename = '/data/sgh/science/2024_minflux/20260106_chromatin_dynamics_all_data.h5'
data       = nl.io.load.hdf5(filename)['data']

# Fits

In [4]:
ct = 'mESC' # all array tracking is in mESC-derived lines; they have this tag
bar = tqdm()

fits = {}
for treatment in ['C36', 'ΔRAD21 (inactive)', 'ΔRAD21 (active)']:

    cond = ['array', ct, treatment]
    fits[treatment] = {
        'single' : {},
        'joints' : {},
    }

    # Conventional
    for dt_tag in ['100ms', '2s']:
        data.makeSelection(['SPT', dt_tag, *cond], logic=all)
        dt = data[0].meta['Δt']
        tau_e = 0.08671 # same exposure for both conditions

        fitdata = data.apply(lambda traj : traj.relative(keepmeta=['MSD', 'Δt']), inplace=False)

        fit = bayesmsd.lib.NPFit(fitdata, motion_blur_f=tau_e, parametrization='(log(αΓ), α)')
        fit.parameters['log(σ²) (dim 1)'].fix_to = 'log(σ²) (dim 0)'
        fit.likelihood_chunksize = 100

        fits[treatment]['single'][f'SPT-{dt_tag}'] = fit

        bar.update()

    # Assemble list of fit(group)s to run
    groups = {
        'SPT 100ms'     : ['SPT-100ms'],
        'SPT 2s'        : ['SPT-2s'],
        'SPT'           : ['SPT-100ms', 'SPT-2s'],
    }

    for groupname in groups:
        fits_dict = fits[treatment]['single']

        fit = bayesmsd.FitGroup({name : fits_dict[name] for name in groups[groupname]})
        fit.parameters['α']       = deepcopy(fits_dict['SPT-100ms'].parameters[      'α (dim 0)'])
        fit.parameters['log(αΓ)'] = deepcopy(fits_dict['SPT-100ms'].parameters['log(αΓ) (dim 0)'])

        # hacky...
        def patch_initial_params(self=fit):
            params = type(self).initial_params(self)
            a    = [val for key, val in params.items() if      'α' in key][0]
            logG = [val for key, val in params.items() if 'log(αΓ)' in key][0]
            params['α'] = a
            params['log(αΓ)'] = logG
            return params
        fit.initial_params = patch_initial_params

        for fitname in fit.fits_dict:
            fit.parameters[fitname+f' α (dim 0)'].fix_to = 'α'
            if fitname == 'minflux':
                fit.parameters[fitname+f' log(αΓ) (dim 0)'].fix_to = 'log(αΓ)'
            else: # not minflux, so correct for 2-loc
                def twoGref(params): return params['log(αΓ)']+np.log(2)
                fit.parameters[fitname+f' log(αΓ) (dim 0)'].fix_to = twoGref

        fits[treatment]['joints'][groupname] = fit

        bar.update()

bar.close()


2it [02:15, 67.89s/it]

1it [00:00,  7.44it/s]
2it [00:00,  7.68it/s]
6it [00:00, 18.18it/s]
11it [00:00, 23.17it/s]
15it [00:00, 22.06it/s]


In [5]:
fitres = {}
for treatment in fits:
    print()
    print(17*'=')
    print(f'|| {ct:>5s} {treatment:<5s} ||')
    print(17*'=')
    print()
    
    fitres[treatment] = {}
    for name in fits[treatment]['joints']:
        print(name)
        print('='*20)

        with nl.Parallelize():
            fitres[treatment][name] = fits[treatment]['joints'][name].run(show_progress=True)

        for key in fitres[treatment][name]['params']:
            print(key, fitres[treatment][name]['params'][key])
        print()


||  mESC C36   ||

SPT 100ms


fit iterations: 57it [00:08,  6.79it/s]


SPT-100ms log(σ²) (dim 0) -6.4739832491722
α 0.47296267150048493
log(αΓ) -5.910119312570625
SPT-100ms α (dim 0) 0.47296267150048493
SPT-100ms log(αΓ) (dim 0) -5.21697213201068

SPT 2s


fit iterations: 59it [00:08,  7.06it/s]


SPT-2s log(σ²) (dim 0) -5.536114206153824
α 0.5994463338862202
log(αΓ) -6.393608528443302
SPT-2s α (dim 0) 0.5994463338862202
SPT-2s log(αΓ) (dim 0) -5.700461347883357

SPT


...

SPT-100ms log(σ²) (dim 0) -6.4578565409946895
α 0.4481629082607
log(αΓ) -5.917858860866568
SPT-100ms α (dim 0) 0.4481629082607
SPT-100ms log(αΓ) (dim 0) -5.224711680306623

SPT 2s


fit iterations: 61it [00:07,  7.66it/s]


SPT-2s log(σ²) (dim 0) -74.51624192310504
α 0.3535016691290125
log(αΓ) -5.856864934150098
SPT-2s α (dim 0) 0.3535016691290125
SPT-2s log(αΓ) (dim 0) -5.163717753590152

SPT


fit iterations: 167it [00:25,  6.53it/s]

SPT-100ms log(σ²) (dim 0) -6.471450461263924
SPT-2s log(σ²) (dim 0) -6.518671664124522
α 0.421951988906213
log(αΓ) -5.978778227897628
SPT-100ms α (dim 0) 0.421951988906213
SPT-2s α (dim 0) 0.421951988906213
SPT-100ms log(αΓ) (dim 0) -5.285631047337683
SPT-2s log(αΓ) (dim 0) -5.285631047337683



In [6]:
nl.io.write.hdf5(fitres, f'/data/sgh/science/2024_minflux/fits/20260111_fitres_NPFit-aGparam_SPT_array.h5')

## Profiler
Estimate credible intervals for point estimates from profile likelihood. __Attention: computationally expensive__

This can also move the point estimate, if we find better parameters while exploring

In [8]:
fitres = nl.io.load.hdf5(f'/data/sgh/science/2024_minflux/fits/20260111_fitres_NPFit-aGparam_SPT_array.h5')
mci = {}
for treatment in fitres:
    print()
    print(17*'=')
    print(f'|| {ct:>5s} {treatment:<5s} ||')
    print(17*'=')
    print()
    
    mci[treatment] = {}
    for name in fits[treatment]['joints']:
        print(name)
        print('='*20)
        
        profiler = bayesmsd.Profiler(fits[treatment]['joints'][name], max_restarts=50)
        profiler.point_estimate = fitres[treatment][name]

        with nl.Parallelize():
            mci[treatment][name] = profiler.find_MCI(show_progress=True)

        for key in mci[treatment][name]:
            m, (cil, cih) = mci[treatment][name][key]
            print(f"{key:>25s} = {m:>6.3f} [{cil:>6.3f}, {cih:>6.3f}]")
        print()


||  mESC C36   ||

SPT 100ms


profiler iterations: 78it [02:46,  2.13s/it]


SPT-100ms log(σ²) (dim 0) = -6.474 [-6.513, -6.437]
                        α =  0.473 [ 0.445,  0.500]
                  log(αΓ) = -5.910 [-5.955, -5.867]

SPT 2s


profiler iterations: 95it [03:45,  2.38s/it]


   SPT-2s log(σ²) (dim 0) = -5.536 [-5.659, -5.436]
                        α =  0.599 [ 0.558,  0.642]
                  log(αΓ) = -6.394 [-6.468, -6.324]

SPT


...

SPT-100ms log(σ²) (dim 0) = -6.544 [-6.582, -6.508]
                        α =  0.446 [ 0.417,  0.475]
                  log(αΓ) = -6.212 [-6.257, -6.166]

SPT 2s


profiler iterations: 93it [03:28,  2.24s/it]


   SPT-2s log(σ²) (dim 0) = -5.751 [-5.931, -5.616]
                        α =  0.569 [ 0.521,  0.616]
                  log(αΓ) = -6.441 [-6.523, -6.350]

SPT


profiler iterations: 174it [19:01,  6.56s/it]

SPT-100ms log(σ²) (dim 0) = -6.532 [-6.562, -6.504]
   SPT-2s log(σ²) (dim 0) = -6.348 [-6.449, -6.258]
                        α =  0.451 [ 0.438,  0.464]
                  log(αΓ) = -6.216 [-6.238, -6.194]



In [9]:
nl.io.write.hdf5(mci, f'/data/sgh/science/2024_minflux/fits/20260111_mci_NPFit-aGparam_SPT_array.h5')